# Projet Kayak — Notebook pur

Pipeline complet : **Nominatim → OpenWeather → Top 5 → Booking → Merge → S3 → Neon → Cartes**

> Ouvrir ce notebook depuis la racine du projet. Le fichier `.env` doit être présent localement.

## 0. Imports & configuration

In [ ]:
# ─── Imports standard Python ───────────────────────────────────────────────
import os           # lecture des variables d'environnement
import math         # calculs trigonométriques pour décaler les marqueurs sur la carte
import time         # pause entre les appels API (respecter les limites de taux)
from pathlib import Path                    # gestion des chemins de fichiers
from urllib.parse import quote_plus, urljoin  # encodage d'URL et construction de liens

# ─── Librairies tierces ─────────────────────────────────────────────────────
import pandas as pd          # manipulation de tableaux de données (DataFrames)
import requests              # appels HTTP vers les APIs externes
import folium                # génération de cartes interactives HTML
from bs4 import BeautifulSoup  # parsing du HTML récupéré depuis Booking
from dotenv import load_dotenv  # lecture du fichier .env (clés API locales)
from IPython.display import IFrame, display  # affichage des cartes dans le notebook

# ─── Chemins du projet ──────────────────────────────────────────────────────
ROOT = Path(".")                          # racine du projet (dossier courant)
DATA_DIR = ROOT / "data"                  # dossier des fichiers CSV
RAW_HTML_DIR = DATA_DIR / "raw_booking_html"  # dossier des pages HTML brutes Booking
MAPS_DIR = ROOT / "maps"                  # dossier des cartes HTML finales

# Création des dossiers s'ils n'existent pas encore
DATA_DIR.mkdir(exist_ok=True)
RAW_HTML_DIR.mkdir(exist_ok=True)
MAPS_DIR.mkdir(exist_ok=True)

# ─── Chargement des clés API ────────────────────────────────────────────────
# override=True : force le rechargement même si les variables sont déjà en mémoire
# Indispensable quand on change une clé dans .env sans redémarrer le kernel
load_dotenv(ROOT / ".env", override=True)
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")  # météo
SCRAPINGBEE_API_KEY = os.getenv("SCRAPINGBEE_API_KEY")  # scraping Booking
NEON_DATABASE_URL   = os.getenv("NEON_DATABASE_URL")    # base PostgreSQL Neon

# Vérification rapide : affiche OK si la clé est présente, MANQUANTE sinon
print("OPENWEATHER_API_KEY :", "OK" if OPENWEATHER_API_KEY else "MANQUANTE")
print("SCRAPINGBEE_API_KEY :", "OK" if SCRAPINGBEE_API_KEY else "MANQUANTE")
print("NEON_DATABASE_URL   :", "OK" if NEON_DATABASE_URL else "MANQUANTE")

## 1. Géocodage Nominatim

In [ ]:
# ─── Liste des 35 villes touristiques françaises à analyser ─────────────────
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle",
]

# En-tête HTTP requis par Nominatim : identifie l'application qui fait la requête
# Sans User-Agent valide, Nominatim peut refuser la requête (politique d'usage)
NOMINATIM_HEADERS = {"User-Agent": "KayakProjectBirane/1.0 (student project geocoding)"}


def geocode_city(city: str) -> tuple:
    """
    Interroge l'API Nominatim (OpenStreetMap) pour obtenir les coordonnées GPS d'une ville.
    Retourne un tuple : (dict_résultat, statut "OK"/"KO", code HTTP, message d'erreur)
    """
    url = "https://nominatim.openstreetmap.org/search"
    # Paramètres : on cherche la ville en France, format JSON, on ne veut qu'un seul résultat
    params = {"q": f"{city}, France", "format": "jsonv2", "limit": 1}
    try:
        response = requests.get(url, params=params, headers=NOMINATIM_HEADERS, timeout=30)
        http_code = response.status_code

        # Si l'API renvoie une erreur HTTP (4xx ou 5xx)
        if response.status_code != 200:
            return {"city": city, "lat": None, "lon": None}, "KO", http_code, f"HTTP {http_code}"

        results = response.json()

        # Nominatim peut répondre 200 mais sans résultat si la ville est inconnue
        if not results:
            return {"city": city, "lat": None, "lon": None}, "KO", http_code, "Aucun résultat retourné"

        # On prend le premier résultat et on extrait latitude et longitude
        first = results[0]
        return {"city": city, "lat": float(first["lat"]), "lon": float(first["lon"])}, "OK", http_code, None

    except requests.exceptions.Timeout:
        return {"city": city, "lat": None, "lon": None}, "KO", None, "Timeout"
    except requests.exceptions.ConnectionError as e:
        return {"city": city, "lat": None, "lon": None}, "KO", None, f"ConnectionError: {e}"
    except Exception as e:
        return {"city": city, "lat": None, "lon": None}, "KO", None, str(e)


# ─── Boucle principale : géocodage des 35 villes ────────────────────────────
rows = []   # liste qui va accumuler les résultats (city, lat, lon)
errors = [] # liste des villes en erreur pour le récapitulatif final

for idx, city in enumerate(CITIES, start=1):
    row, status, http_code, error_msg = geocode_city(city)
    rows.append(row)

    code_str = f"HTTP {http_code}" if http_code else "N/A"
    if status == "OK":
        print(f"[{idx:02d}/{len(CITIES)}] {status} ({code_str}) — {city} -> lat={row['lat']:.4f}, lon={row['lon']:.4f}")
    else:
        print(f"[{idx:02d}/{len(CITIES)}] {status} ({code_str}) — {city} -> ERREUR : {error_msg}")
        errors.append({"city": city, "http_code": http_code, "error": error_msg})

    # Pause obligatoire : Nominatim impose max 1 requête/seconde pour les usages gratuits
    time.sleep(1.1)

# ─── Sauvegarde du résultat ──────────────────────────────────────────────────
df_geo = pd.DataFrame(rows)
df_geo.insert(0, "id", range(1, len(df_geo) + 1))  # colonne id numérique en première position
df_geo.to_csv(DATA_DIR / "cities_geocoded.csv", index=False, encoding="utf-8")

# ─── Récapitulatif ───────────────────────────────────────────────────────────
print(f"\n{'='*55}")
ok_count = sum(1 for r in rows if r["lat"] is not None)
print(f"Résultat : {ok_count}/{len(CITIES)} villes géocodées avec succès")
if errors:
    print(f"Erreurs ({len(errors)}) :")
    for e in errors:
        print(f"  - {e['city']} | code={e['http_code']} | {e['error']}")
else:
    print("Aucune erreur.")
print(f"{'='*55}")
print("Saved -> data/cities_geocoded.csv")
df_geo.head()

## 2. Météo OpenWeather + calcul du score

In [ ]:
from collections import defaultdict  # dictionnaire avec valeur par défaut, pratique pour grouper par jour

# ─── Formule du score météo ──────────────────────────────────────────────────
def compute_weather_score(avg_temp_7d, avg_pop_7d, total_rain_7d, avg_wind_7d) -> float:
    """
    Calcule un score météo sur 100 à partir de 4 indicateurs.
    Logique métier du projet : on part de 100 et on pénalise les mauvaises conditions.
      - Température : on pénalise l'écart à 24°C (température idéale pour le kayak)
      - Probabilité de pluie (pop) : forte pénalité car pluie = conditions dangereuses
      - Pluie totale : pénalité proportionnelle au cumul de précipitations
      - Vent : légère pénalité (vent fort = navigation difficile)
    """
    score = 100.0
    score -= abs(avg_temp_7d - 24) * 2.0   # -2 pts par degré d'écart à 24°C
    score -= avg_pop_7d * 30.0              # -30 pts si probabilité de pluie = 100%
    score -= total_rain_7d * 1.5            # -1.5 pt par mm de pluie cumulée
    score -= avg_wind_7d * 0.8             # -0.8 pt par m/s de vent moyen
    return round(score, 2)


def fetch_forecast(lat: float, lon: float) -> tuple:
    """
    Appelle l'endpoint gratuit OpenWeather /forecast.
    Retourne des prévisions toutes les 3h sur 5 jours (40 créneaux au total).
    Retourne : (données JSON, code HTTP, message d'erreur)

    Note : on utilise /forecast (plan gratuit) et non /onecall (plan payant).
    """
    if not OPENWEATHER_API_KEY:
        return None, None, "OPENWEATHER_API_KEY manquante dans .env"

    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",  # températures en °C
        "cnt": 40,          # 5 jours × 8 créneaux de 3h = 40 entrées
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        http_code = response.status_code
        if response.status_code != 200:
            return None, http_code, f"HTTP {http_code} — {response.json().get('message', '')}"
        return response.json(), http_code, None
    except requests.exceptions.Timeout:
        return None, None, "Timeout"
    except requests.exceptions.ConnectionError as e:
        return None, None, f"ConnectionError: {e}"
    except Exception as e:
        return None, None, str(e)


def summarize_city_weather(city: str, lat: float, lon: float) -> tuple:
    """
    Transforme les 40 créneaux de 3h en indicateurs journaliers moyens,
    puis calcule le score météo final.
    Retourne : (dict_résultat, statut "OK"/"KO", code HTTP, message d'erreur)
    """
    data, http_code, error_msg = fetch_forecast(lat, lon)
    if data is None:
        return None, "KO", http_code, error_msg

    # Regroupement des créneaux par jour (clé = "YYYY-MM-DD")
    # defaultdict évite d'initialiser manuellement chaque clé
    daily = defaultdict(lambda: {"temps": [], "pops": [], "rains": [], "winds": []})
    for slot in data.get("list", []):
        day_key = slot["dt_txt"][:10]  # on garde seulement la date, pas l'heure
        daily[day_key]["temps"].append(slot["main"]["temp"])
        daily[day_key]["pops"].append(slot.get("pop", 0))               # probabilité de pluie (0 à 1)
        daily[day_key]["rains"].append(slot.get("rain", {}).get("3h", 0))  # mm de pluie sur 3h
        daily[day_key]["winds"].append(slot["wind"]["speed"])            # vitesse vent en m/s

    # On prend les 5 premiers jours triés chronologiquement
    days = sorted(daily.keys())[:5]

    if len(days) < 3:
        return None, "KO", http_code, f"Pas assez de jours de prévision ({len(days)})"

    # Calcul des moyennes et totaux sur les 5 jours
    avg_temp   = round(sum(sum(daily[d]["temps"]) / len(daily[d]["temps"]) for d in days) / len(days), 2)
    avg_pop    = round(sum(sum(daily[d]["pops"])  / len(daily[d]["pops"])  for d in days) / len(days), 4)
    total_rain = round(sum(sum(daily[d]["rains"]) for d in days), 2)  # cumul total, pas une moyenne
    avg_wind   = round(sum(sum(daily[d]["winds"]) / len(daily[d]["winds"]) for d in days) / len(days), 2)

    result = {
        "city": city, "lat": lat, "lon": lon,
        "avg_temp_7d": avg_temp,
        "avg_pop_7d": avg_pop,
        "total_rain_7d": total_rain,
        "avg_wind_7d": avg_wind,
        "weather_score": compute_weather_score(avg_temp, avg_pop, total_rain, avg_wind),
    }
    return result, "OK", http_code, None


# ─── Boucle principale : météo des 35 villes ────────────────────────────────
df_geo = pd.read_csv(DATA_DIR / "cities_geocoded.csv")  # on repart du fichier géocodé
weather_rows = []
errors = []

for idx, row in df_geo.iterrows():
    city = row["city"]

    # On ignore les villes sans coordonnées GPS (échec éventuel du géocodage)
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        print(f"[{idx+1:02d}/{len(df_geo)}] SKIP — {city} (pas de coordonnées GPS)")
        continue

    result, status, http_code, error_msg = summarize_city_weather(city, float(row["lat"]), float(row["lon"]))
    code_str = f"HTTP {http_code}" if http_code else "N/A"

    if status == "OK":
        weather_rows.append(result)
        print(f"[{idx+1:02d}/{len(df_geo)}] OK  ({code_str}) — {city} | score={result['weather_score']} | temp={result['avg_temp_7d']}°C")
    else:
        print(f"[{idx+1:02d}/{len(df_geo)}] KO  ({code_str}) — {city} -> ERREUR : {error_msg}")
        errors.append({"city": city, "http_code": http_code, "error": error_msg})

# ─── Sauvegarde ──────────────────────────────────────────────────────────────
df_weather = pd.DataFrame(weather_rows)
df_weather.to_csv(DATA_DIR / "weather.csv", index=False, encoding="utf-8")

print(f"\n{'='*60}")
print(f"Résultat : {len(weather_rows)}/{len(df_geo)} villes météo récupérées avec succès")
if errors:
    print(f"Erreurs ({len(errors)}) :")
    for e in errors:
        print(f"  - {e['city']} | code={e['http_code']} | {e['error']}")
else:
    print("Aucune erreur.")
print(f"{'='*60}")
print("Saved -> data/weather.csv")
df_weather[["city", "avg_temp_7d", "avg_pop_7d", "total_rain_7d", "avg_wind_7d", "weather_score"]].head(10)

## 3. Sélection du Top 5

In [ ]:
# ─── Sélection des 5 meilleures destinations ────────────────────────────────
df_weather = pd.read_csv(DATA_DIR / "weather.csv")

# Tri décroissant par score météo, on ne garde que les 5 premières lignes
# C'est ce fichier qui pilote toute la suite : seules ces 5 villes seront scrapées
df_top = (
    df_weather
    .sort_values(by="weather_score", ascending=False)
    .head(5)
    .copy()  # .copy() pour éviter de modifier le DataFrame original par inadvertance
)

df_top.to_csv(DATA_DIR / "top_cities.csv", index=False, encoding="utf-8")

print("Saved -> data/top_cities.csv")
df_top[["city", "weather_score"]]

## 4. Scraping Booking (ScrapingBee)

In [ ]:
# ─── Test de la clé ScrapingBee avant le scraping complet ───────────────────
# On recharge le .env pour être sûr d'utiliser la clé la plus récente
load_dotenv(ROOT / ".env", override=True)
SCRAPINGBEE_API_KEY = os.getenv("SCRAPINGBEE_API_KEY")
print(f"Clé chargée : {SCRAPINGBEE_API_KEY[:10]}...{SCRAPINGBEE_API_KEY[-6:]}")

# On envoie une vraie requête Booking avec les mêmes paramètres que le scraping réel
# render_js=true : demande à ScrapingBee d'exécuter le JavaScript de la page (coûte 5 crédits)
# stealth_proxy=true : utilise un proxy discret pour éviter d'être bloqué par Booking
test_url = "https://www.booking.com/searchresults.fr.html?ss=Paris"
params_test = {
    "api_key": SCRAPINGBEE_API_KEY,
    "url": test_url,
    "render_js": "true",
    "stealth_proxy": "true",
    "country_code": "fr",   # on simule un navigateur basé en France
    "block_resources": "false",
}

try:
    r = requests.get("https://app.scrapingbee.com/api/v1/", params=params_test, timeout=90)
    http_code = r.status_code
    if http_code == 200:
        # Les headers de réponse ScrapingBee indiquent le coût et le solde restant
        credits_used = r.headers.get("Spb-cost", "?")
        credits_left = r.headers.get("Spb-remaining-api-credits", "?")
        # On compte les property-cards pour vérifier que la page contient bien des hôtels
        cards = r.text.count('data-testid="property-card"')
        print(f"[SCRAPINGBEE] OK (HTTP 200)")
        print(f"  Crédits utilisés : {credits_used} | Crédits restants : {credits_left}")
        print(f"  property-cards détectées : {cards}")
    else:
        try:
            detail = r.json()
        except Exception:
            detail = r.text[:300]
        print(f"[SCRAPINGBEE] KO (HTTP {http_code}) — {detail}")
except requests.exceptions.Timeout:
    print("[SCRAPINGBEE] KO — Timeout")
except Exception as e:
    print(f"[SCRAPINGBEE] KO — {e}")

In [ ]:
def build_booking_url_variants(city: str) -> list:
    """
    Construit 4 variantes d'URL Booking pour une même ville.
    Pourquoi plusieurs variantes ? Booking ne répond pas de façon identique selon la forme de l'URL.
    On essaiera les variantes dans l'ordre jusqu'à trouver une page avec des hôtels.
    quote_plus() encode les espaces et accents pour les rendre valides dans une URL.
    """
    city_simple = quote_plus(city)                    # ex: "Aix+en+Provence"
    city_france = quote_plus(f"{city}, France")       # ex: "Aix+en+Provence%2C+France"
    return [
        ("city_france",      f"https://www.booking.com/searchresults.fr.html?ss={city_france}"),
        ("city_full",        f"https://www.booking.com/searchresults.fr.html?ss={city_simple}&lang=fr&group_adults=2&no_rooms=1&group_children=0"),
        ("city_simple",      f"https://www.booking.com/searchresults.fr.html?ss={city_simple}"),
        ("city_france_full", f"https://www.booking.com/searchresults.fr.html?ss={city_france}&lang=fr&group_adults=2&no_rooms=1&group_children=0"),
    ]


def fetch_html_scrapingbee(target_url: str) -> tuple:
    """
    Envoie l'URL Booking à ScrapingBee qui se charge de récupérer le HTML rendu.
    On utilise ScrapingBee car Booking bloque les requêtes Python directes (détection de bot).
    Retourne : (html, code HTTP, message d'erreur) — ne lève jamais d'exception.
    """
    if not SCRAPINGBEE_API_KEY:
        return None, None, "SCRAPINGBEE_API_KEY manquante dans .env"
    params = {
        "api_key": SCRAPINGBEE_API_KEY,
        "url": target_url,
        "stealth_proxy": "true",     # proxy furtif pour contourner la détection Booking
        "country_code": "fr",        # simule un utilisateur français
        "render_js": "true",         # exécute le JavaScript (nécessaire pour Booking)
        "block_resources": "false",  # on ne bloque pas les ressources (images, CSS...)
    }
    try:
        response = requests.get("https://app.scrapingbee.com/api/v1/", params=params, timeout=90)
        http_code = response.status_code
        if response.status_code != 200:
            try:
                detail = response.json().get("message", response.text[:200])
            except Exception:
                detail = response.text[:200]
            return None, http_code, f"HTTP {http_code} — {detail}"
        return response.text, http_code, None
    except requests.exceptions.Timeout:
        return None, None, "Timeout"
    except requests.exceptions.ConnectionError as e:
        return None, None, f"ConnectionError: {e}"
    except Exception as e:
        return None, None, str(e)


def html_has_property_cards(html: str):
    """
    Vérifie si le HTML contient de vraies fiches hôtels Booking.
    Booking utilise l'attribut data-testid="property-card" sur chaque hôtel.
    Si ce marqueur est absent, la page est générique (accueil, captcha, etc.)
    """
    soup = BeautifulSoup(html, "lxml")
    title = soup.title.get_text(" ", strip=True) if soup.title else "NO TITLE"
    count = html.count('data-testid="property-card"')
    return count > 0, count, title


def save_city_html(city: str, html: str) -> Path:
    """
    Sauvegarde le HTML brut dans data/raw_booking_html/.
    On normalise le nom de fichier (minuscules, sans accents, espaces → underscores).
    """
    safe = (
        city.lower()
        .replace(" ", "_").replace("é", "e").replace("è", "e").replace("ê", "e")
        .replace("à", "a").replace("ù", "u").replace("î", "i").replace("ï", "i")
        .replace("ô", "o").replace("ç", "c").replace("'", "").replace("-", "_")
    )
    path = RAW_HTML_DIR / f"booking_{safe}.html"
    path.write_text(html, encoding="utf-8")
    return path


# ─── Boucle principale : scraping des 5 villes du top ───────────────────────
df_top = pd.read_csv(DATA_DIR / "top_cities.csv")
errors = []

for idx, row in df_top.iterrows():
    city = row["city"]
    print(f"\n[BOOKING] {idx + 1}/{len(df_top)} -> {city}")
    variants = build_booking_url_variants(city)
    html_saved, used_variant = None, None

    # On teste les variantes d'URL dans l'ordre jusqu'à trouver une page valide
    for label, url in variants:
        print(f"  [TRY] variant={label}")
        html, http_code, error_msg = fetch_html_scrapingbee(url)
        code_str = f"HTTP {http_code}" if http_code else "N/A"

        if html is None:
            print(f"  [KO]  ({code_str}) variant={label} -> {error_msg}")
            # Erreur bloquante (clé invalide, accès refusé) : inutile d'essayer les autres variantes
            if http_code in (401, 403) or http_code is None:
                errors.append({"city": city, "variant": label, "http_code": http_code, "error": error_msg})
                break
            continue

        ok, count, title = html_has_property_cards(html)
        print(f"  [OK]  ({code_str}) variant={label} | property_cards={count} | title={title}")

        if ok:
            # On a trouvé une page avec des hôtels, on arrête d'essayer les autres variantes
            html_saved = html
            used_variant = label
            break
        else:
            print(f"  [SKIP] variant={label} — page sans résultats hôtels")

    if html_saved is None:
        msg = f"Aucune variante valide pour {city}"
        print(f"  [WARN] {msg}")
        errors.append({"city": city, "variant": "all", "http_code": None, "error": msg})
        continue

    saved_path = save_city_html(city, html_saved)
    print(f"  [SAVED] {saved_path} (variant={used_variant})")

# ─── Récapitulatif ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
ok_count = len(df_top) - len({e["city"] for e in errors})
print(f"Résultat : {ok_count}/{len(df_top)} villes scrapées avec succès")
if errors:
    print(f"Erreurs ({len(errors)}) :")
    for e in errors:
        print(f"  - {e['city']} | variant={e['variant']} | code={e['http_code']} | {e['error']}")
else:
    print("Aucune erreur.")
print(f"{'='*60}")

## 5. Parsing Booking → CSV hôtels

In [ ]:
# Correspondance entre nom de fichier HTML et nom de ville propre
# Nécessaire car le nom de fichier est normalisé (sans accents, underscores)
# alors que le nom de ville doit rester exact pour la jointure avec la météo
FILENAME_TO_CITY = {
    "booking_avignon.html":            "Avignon",
    "booking_aix_en_provence.html":    "Aix en Provence",
    "booking_nimes.html":              "Nimes",
    "booking_bormes_les_mimosas.html": "Bormes les Mimosas",
    "booking_strasbourg.html":         "Strasbourg",
}


def extract_score_from_card(card) -> float | None:
    """
    Extrait la note client (0-10) d'une fiche hôtel Booking.
    On cherche d'abord le sélecteur principal, puis des sélecteurs alternatifs
    car Booking peut changer la structure HTML selon la version de la page.
    """
    # Sélecteur principal : balise avec data-testid="review-score"
    score_tag = card.select_one('[data-testid="review-score"]')
    if score_tag:
        text = score_tag.get_text(" ", strip=True).replace(",", ".")  # virgule → point pour float()
        for token in text.split():
            try:
                v = float(token)
                if 0 <= v <= 10:  # filtre pour ne garder que des notes valides
                    return v
            except Exception:
                pass

    # Sélecteurs alternatifs si la structure principale n'est pas trouvée
    for tag in card.select('[data-testid="review-score"] div, [aria-label*="Note"], [aria-label*="Scored"]'):
        text = tag.get_text(" ", strip=True).replace(",", ".")
        for token in text.split():
            try:
                v = float(token)
                if 0 <= v <= 10:
                    return v
            except Exception:
                pass
    return None  # pas de note trouvée


def extract_name_from_card(card) -> str | None:
    """
    Extrait le nom de l'hôtel depuis la fiche Booking.
    On essaie plusieurs sélecteurs CSS car Booking utilise des classes CSS qui peuvent varier.
    En dernier recours, on utilise le texte du lien vers la page de l'hôtel.
    """
    for sel in ('[data-testid="title"]', "div.f6431b446c", "div[data-testid='title'] div"):
        tag = card.select_one(sel)
        if tag:
            txt = tag.get_text(" ", strip=True)
            if txt and len(txt) > 2:  # filtre les textes vides ou trop courts
                return txt
    # Fallback : lien vers la page hôtel
    link = card.select_one('a[href*="/hotel/"]')
    if link:
        txt = link.get_text(" ", strip=True)
        if txt and len(txt) > 2:
            return txt
    return None


def extract_link_from_card(card) -> str | None:
    """
    Extrait l'URL de la page hôtel sur Booking.
    urljoin() reconstruit l'URL complète si le href est relatif (commence par /).
    """
    link = card.select_one('a[href*="/hotel/"]')
    if not link:
        return None
    href = link.get("href")
    return urljoin("https://www.booking.com", href) if href else None


def extract_description_from_card(card) -> str | None:
    """
    Extrait la description courte de la chambre/logement proposé.
    Ces sélecteurs correspondent aux blocs de configuration de chambre sur Booking.
    """
    for sel in (
        '[data-testid="property-card-unit-configuration"]',
        '[data-testid="property-card-unit-configuration-group"]',
        'div.abf093bdfe',
        'div.c624d7469d',
    ):
        tag = card.select_one(sel)
        if tag:
            txt = tag.get_text(" ", strip=True)
            if txt and len(txt) > 10:
                return txt
    return None


def parse_booking_hotels(html: str, city: str, max_hotels: int = 10) -> list:
    """
    Parcourt le HTML d'une page Booking et extrait les informations des hôtels.
    On ne garde que les vraies property-cards (pas les publicités ou suggestions).
    seen_urls évite les doublons si un même hôtel apparaît plusieurs fois.
    """
    soup = BeautifulSoup(html, "lxml")
    # Sélection de toutes les fiches hôtels (chaque hôtel = une div property-card)
    cards = soup.select('div[data-testid="property-card"]')
    print(f"  [PARSE] {city} — {len(cards)} property-cards trouvées")

    rows = []
    seen = set()  # ensemble des URLs déjà vues pour éviter les doublons

    for card in cards:
        name = extract_name_from_card(card)
        url  = extract_link_from_card(card)

        # On ignore les fiches sans nom ou sans lien, et les doublons
        if not name or not url or url in seen:
            continue
        seen.add(url)

        rows.append({
            "city": city,
            "hotel_name": name,
            "hotel_score": extract_score_from_card(card),
            "url": url,
            "description": extract_description_from_card(card),
        })

        # On s'arrête dès qu'on a atteint le nombre d'hôtels voulu
        if len(rows) >= max_hotels:
            break

    return rows


# ─── Boucle principale : parsing de tous les fichiers HTML ───────────────────
all_rows = []

for path in sorted(RAW_HTML_DIR.glob("booking_*.html")):
    # Récupère le nom de ville depuis le dictionnaire, ou génère un nom approximatif
    city = FILENAME_TO_CITY.get(
        path.name,
        path.stem.replace("booking_", "").replace("_", " ").title()
    )
    print(f"\n[FILE] {path.name} -> {city}")

    # Lecture du fichier HTML sauvegardé lors du scraping
    html = path.read_text(encoding="utf-8", errors="ignore")
    rows = parse_booking_hotels(html, city=city)
    print(f"  -> {len(rows)} hôtels extraits")
    all_rows.extend(rows)

# ─── Sauvegarde du CSV hôtels ────────────────────────────────────────────────
df_hotels = pd.DataFrame(all_rows)
df_hotels.to_csv(DATA_DIR / "hotels_multi_cities.csv", index=False, encoding="utf-8")

print("\nSaved -> data/hotels_multi_cities.csv")
print(df_hotels["city"].value_counts())
df_hotels.head()

## 6. Fusion hôtels + météo → dataset final

In [ ]:
# ─── Fusion météo + hôtels ───────────────────────────────────────────────────
df_weather = pd.read_csv(DATA_DIR / "weather.csv")
df_hotels  = pd.read_csv(DATA_DIR / "hotels_multi_cities.csv")

# On ne garde que les données météo des villes qui ont été effectivement scrapées
# (au cas où le scraping n'aurait pas fonctionné pour toutes les villes du top 5)
cities_scraped = set(df_hotels["city"].dropna().unique())
df_weather_top = df_weather[df_weather["city"].isin(cities_scraped)].copy()

# Jointure gauche sur la colonne "city" :
# chaque hôtel hérite des indicateurs météo de sa ville
# how="left" : on garde tous les hôtels même si la météo manque pour une ville
df_final = df_hotels.merge(df_weather_top, on="city", how="left")
df_final.to_csv(DATA_DIR / "final_kayak_results.csv", index=False, encoding="utf-8")

print("Saved -> data/final_kayak_results.csv")
print("Shape :", df_final.shape)  # doit afficher (50, 12) si tout s'est bien passé
print(df_final["city"].value_counts())
df_final.head()

## 7. Chargement dans Neon / PostgreSQL

In [ ]:
import psycopg2                      # driver Python pour PostgreSQL
from psycopg2.extras import execute_values  # insertion batch (plus rapide qu'un INSERT par ligne)

# Colonnes attendues dans le CSV final — doit correspondre exactement à la table SQL
EXPECTED_COLUMNS = [
    "city", "hotel_name", "hotel_score", "url", "description",
    "lat", "lon", "avg_temp_7d", "avg_pop_7d", "total_rain_7d",
    "avg_wind_7d", "weather_score",
]

if not NEON_DATABASE_URL:
    print("NEON_DATABASE_URL manquante — étape ignorée")
else:
    df_load = pd.read_csv(DATA_DIR / "final_kayak_results.csv")[EXPECTED_COLUMNS].copy()

    # Conversion des NaN pandas en None Python (psycopg2 ne comprend pas les NaN)
    rows = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in df_load.itertuples(index=False, name=None)
    ]

    # Connexion à la base Neon (PostgreSQL hébergé dans le cloud)
    conn = psycopg2.connect(NEON_DATABASE_URL)
    conn.autocommit = False  # on gère la transaction manuellement

    try:
        with conn.cursor() as cur:
            # On recrée la table à chaque exécution pour repartir d'une base propre
            # DROP IF EXISTS évite une erreur si la table n'existe pas encore
            cur.execute("""
                DROP TABLE IF EXISTS kayak_results;
                CREATE TABLE kayak_results (
                    city TEXT,
                    hotel_name TEXT,
                    hotel_score DOUBLE PRECISION,
                    url TEXT,
                    description TEXT,
                    lat DOUBLE PRECISION,
                    lon DOUBLE PRECISION,
                    avg_temp_7d DOUBLE PRECISION,
                    avg_pop_7d DOUBLE PRECISION,
                    total_rain_7d DOUBLE PRECISION,
                    avg_wind_7d DOUBLE PRECISION,
                    weather_score DOUBLE PRECISION
                );
            """)

            # execute_values insère toutes les lignes en un seul appel SQL (beaucoup plus rapide)
            execute_values(
                cur,
                """INSERT INTO kayak_results
                   (city, hotel_name, hotel_score, url, description,
                    lat, lon, avg_temp_7d, avg_pop_7d, total_rain_7d, avg_wind_7d, weather_score)
                   VALUES %s""",
                rows,
                page_size=100,  # envoi par lots de 100 lignes
            )
            conn.commit()  # valide la transaction — les données sont maintenant persistées

        print(f"[NEON] Table kayak_results chargée avec succès — {len(rows)} lignes insérées")

    except Exception:
        conn.rollback()  # en cas d'erreur, on annule tout pour ne pas laisser la table en état partiel
        raise

    finally:
        conn.close()  # fermeture de la connexion dans tous les cas

## 8. Génération des cartes Folium

In [ ]:
def clean_text(x) -> str:
    """Convertit une valeur en chaîne propre, renvoie "" si la valeur est NaN."""
    return "" if pd.isna(x) else str(x).strip()

def shorten(text, max_len=42) -> str:
    """Tronque un texte trop long avec '...' pour l'affichage dans le panneau latéral."""
    text = clean_text(text)
    return text if len(text) <= max_len else text[:max_len - 3] + "..."

def prepare_visual_coordinates(df):
    """
    Les hôtels d'une même ville partagent les mêmes coordonnées GPS.
    Pour les rendre tous visibles sur la carte, on les dispose en cercle
    autour du centre de la ville avec un léger décalage géographique.
    Formule trigonométrique : chaque hôtel est placé à un angle régulier sur le cercle.
    """
    parts = []
    for city, group in df.groupby("city", sort=False):
        g = group.copy().reset_index(drop=True)
        base_lat = float(g["lat"].iloc[0])  # coordonnées du centre de la ville
        base_lon = float(g["lon"].iloc[0])
        n = len(g)  # nombre d'hôtels dans cette ville

        for i in range(n):
            if n == 1:
                # Un seul hôtel : pas besoin de décalage
                g.loc[i, "plot_lat"] = base_lat
                g.loc[i, "plot_lon"] = base_lon
            else:
                # Disposition en cercle : angle = i * 360° / n
                angle = (2 * math.pi * i) / n
                radius = 0.025 + (0.004 * (i % 3))  # rayon variable pour éviter la superposition
                g.loc[i, "plot_lat"] = base_lat + radius * math.sin(angle)
                # Division par cos(lat) pour compenser la déformation des longitudes aux latitudes élevées
                g.loc[i, "plot_lon"] = base_lon + (radius * math.cos(angle)) / max(math.cos(math.radians(base_lat)), 0.35)
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


# ════════════════════════════════════════════════════════════
# CARTE 1 — Top 5 destinations (points bleus, une par ville)
# ════════════════════════════════════════════════════════════
df_top5 = pd.read_csv(DATA_DIR / "top_cities.csv")

# Conversion en numérique par sécurité (les CSV stockent parfois les nombres en string)
for col in ["lat", "lon", "weather_score", "avg_temp_7d", "avg_pop_7d", "total_rain_7d", "avg_wind_7d"]:
    df_top5[col] = pd.to_numeric(df_top5[col], errors="coerce")

df_top5 = df_top5.sort_values("weather_score", ascending=False).head(5)

# Centre la carte sur la moyenne des coordonnées des 5 villes
m1 = folium.Map(location=[df_top5["lat"].mean(), df_top5["lon"].mean()], zoom_start=6)
m1.get_root().html.add_child(folium.Element('<h3 align="center" style="font-size:20px;"><b>Top 5 destinations</b></h3>'))

for _, row in df_top5.iterrows():
    # Popup : fenêtre qui s'affiche au clic sur le marqueur
    popup_html = (
        f"<b>{row['city']}</b><br>"
        f"Score : {row['weather_score']:.2f}<br>"
        f"Temp. : {row['avg_temp_7d']:.1f} °C<br>"
        f"Pluie : {row['total_rain_7d']:.1f} mm<br>"
        f"Vent : {row['avg_wind_7d']:.1f} m/s"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=10,
        color="blue", fill=True, fill_color="blue", fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row["city"]  # texte au survol de la souris
    ).add_to(m1)

# fit_bounds ajuste automatiquement le zoom pour que tous les marqueurs soient visibles
m1.fit_bounds([[r["lat"], r["lon"]] for _, r in df_top5.iterrows()], padding=(40, 40))
m1.save(str(MAPS_DIR / "map_top_5_destinations.html"))
print("Saved -> maps/map_top_5_destinations.html")


# ════════════════════════════════════════════════════════════
# CARTE 2 — Top 20 hôtels (points colorés par ville + panneau liste)
# ════════════════════════════════════════════════════════════
# Couleur différente par ville pour distinguer visuellement les clusters
COLOR_MAP = {
    "Aix en Provence": "blue",
    "Avignon": "green",
    "Bormes les Mimosas": "red",
    "Nimes": "purple",
    "Strasbourg": "orange",
}

df_final = pd.read_csv(DATA_DIR / "final_kayak_results.csv")
df_final["lat"] = pd.to_numeric(df_final["lat"], errors="coerce")
df_final["lon"] = pd.to_numeric(df_final["lon"], errors="coerce")
df_final["hotel_score"] = pd.to_numeric(df_final["hotel_score"], errors="coerce").fillna(0)

# Sélection des 20 meilleurs hôtels par note client
df_top20 = df_final.sort_values("hotel_score", ascending=False).head(20).reset_index(drop=True)
df_top20["rank"] = range(1, len(df_top20) + 1)  # numérotation 1 à 20

# Calcul des coordonnées visuelles décalées (cercle autour de la ville)
df_top20 = prepare_visual_coordinates(df_top20)

m2 = folium.Map(location=[df_top20["plot_lat"].mean(), df_top20["plot_lon"].mean()], zoom_start=6)
m2.get_root().html.add_child(folium.Element('<h3 align="center" style="font-size:20px;"><b>Top 20 hôtels</b></h3>'))

for _, row in df_top20.iterrows():
    city  = clean_text(row["city"])
    hotel = clean_text(row["hotel_name"])
    color = COLOR_MAP.get(city, "cadetblue")  # couleur par défaut si ville inconnue
    rank  = int(row["rank"])
    popup_html = (
        f"<b>#{rank} {hotel}</b><br>"
        f"Ville : {city}<br>"
        f"Score : {row['hotel_score']}<br>"
        f"<a href='{clean_text(row['url'])}' target='_blank'>Voir sur Booking</a>"
    )
    # Marqueur cercle coloré
    folium.CircleMarker(
        location=[row["plot_lat"], row["plot_lon"]],
        radius=10,
        color=color, fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{rank}. {hotel}"
    ).add_to(m2)
    # Numéro de rang affiché sur le marqueur (DivIcon = HTML personnalisé)
    folium.Marker(
        location=[row["plot_lat"], row["plot_lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;font-weight:bold;text-align:center;'
                 f'width:18px;height:18px;line-height:18px;background:white;'
                 f'border:1px solid black;border-radius:50%;">{rank}</div>'
        )
    ).add_to(m2)

# Légende des couleurs (HTML injecté directement dans la carte)
legend_html = """
<div style="position:fixed;bottom:35px;left:35px;width:220px;background:white;
    border:2px solid grey;z-index:9999;font-size:14px;padding:10px;">
    <b>Légende</b><br>
    <span style="color:blue;">●</span> Aix en Provence<br>
    <span style="color:green;">●</span> Avignon<br>
    <span style="color:red;">●</span> Bormes les Mimosas<br>
    <span style="color:purple;">●</span> Nimes<br>
    <span style="color:orange;">●</span> Strasbourg
</div>"""
m2.get_root().html.add_child(folium.Element(legend_html))

# Panneau latéral listant les 20 hôtels (HTML fixe en haut à droite de la carte)
list_items = "".join(
    f'<li style="margin-bottom:8px;"><b>{shorten(r["hotel_name"])}</b><br>'
    f'<span style="color:#555;">{r["city"]} – {r["hotel_score"]}</span></li>'
    for _, r in df_top20.sort_values("rank").iterrows()
)
panel_html = f"""
<div style="position:fixed;top:90px;right:18px;width:320px;max-height:72vh;overflow-y:auto;
    background:rgba(255,255,255,0.96);border:1px solid #999;z-index:9999;
    padding:12px;font-size:13px;border-radius:6px;">
    <h4 style="margin:0 0 10px 0;">Top 20 hôtels</h4>
    <ol style="padding-left:22px;margin:0;">{list_items}</ol>
</div>"""
m2.get_root().html.add_child(folium.Element(panel_html))

m2.fit_bounds([[r["plot_lat"], r["plot_lon"]] for _, r in df_top20.iterrows()], padding=(40, 40))
m2.save(str(MAPS_DIR / "map_top_20_hotels.html"))
print("Saved -> maps/map_top_20_hotels.html")

## 9. Affichage des cartes

In [31]:
map1 = MAPS_DIR / "map_top_5_destinations.html"
if map1.exists():
    display(IFrame(src=str(map1), width=1200, height=500))
else:
    print("Carte Top 5 introuvable :", map1)

In [32]:
map2 = MAPS_DIR / "map_top_20_hotels.html"
if map2.exists():
    display(IFrame(src=str(map2), width=1200, height=750))
else:
    print("Carte Top 20 hôtels introuvable :", map2)

---

# Synthèse du projet Kayak

## Contexte

L'entreprise **Kayak** souhaite recommander à ses utilisateurs les meilleures destinations de voyage en France pour pratiquer des activités nautiques. L'objectif est de construire un pipeline de données automatisé qui combine météo et offres hôtelières pour produire un classement objectif.

---

## Pipeline complet

```
Nominatim (GPS)  →  OpenWeather (météo)  →  Score météo  →  Top 5 villes
      ↓                                                          ↓
 35 villes                                             Booking (hôtels)
 géocodées                                                       ↓
                                                        Parsing HTML
                                                                 ↓
                                                    Fusion météo + hôtels
                                                                 ↓
                                                  Stockage S3  +  Neon SQL
                                                                 ↓
                                                       Cartes interactives
```

---

## Étapes détaillées

### 1. Géocodage — Nominatim (OpenStreetMap)
- **Entrée** : liste de 35 villes françaises
- **Traitement** : appel à l'API Nominatim pour obtenir les coordonnées GPS (latitude, longitude)
- **Sortie** : `cities_geocoded.csv`
- **Point technique** : respect de la politique d'usage (1 requête/seconde max, User-Agent obligatoire)

### 2. Météo — OpenWeather `/forecast`
- **Entrée** : coordonnées GPS des 35 villes
- **Traitement** : récupération des prévisions toutes les 3h sur 5 jours, agrégation par jour
- **Indicateurs calculés** : température moyenne, probabilité de pluie, cumul de pluie, vitesse du vent
- **Sortie** : `weather.csv`
- **Point technique** : utilisation de l'endpoint gratuit `/forecast` (le `/onecall` est payant)

### 3. Score météo
La formule pénalise les mauvaises conditions à partir de 100 points :

| Indicateur | Pénalité |
|---|---|
| Écart à 24°C (température idéale kayak) | −2 pts / degré |
| Probabilité de pluie (0 à 1) | −30 pts si 100% |
| Pluie totale cumulée | −1,5 pt / mm |
| Vitesse du vent | −0,8 pt / m/s |

### 4. Sélection du Top 5
- Tri décroissant par `weather_score`
- Seules ces 5 villes sont envoyées au scraping Booking
- **Sortie** : `top_cities.csv`

### 5. Scraping Booking — ScrapingBee
- **Pourquoi ScrapingBee ?** Booking détecte et bloque les requêtes Python directes. ScrapingBee agit comme un navigateur réel avec proxy furtif et rendu JavaScript.
- **Stratégie multi-variantes** : 4 formes d'URL testées dans l'ordre pour chaque ville, on garde la première qui retourne des fiches hôtels
- **Sortie** : 5 fichiers HTML dans `data/raw_booking_html/`

### 6. Parsing HTML — BeautifulSoup
- Extraction des fiches identifiées par `data-testid="property-card"`
- Pour chaque hôtel : nom, note client, URL, description
- 10 hôtels maximum par ville → **50 hôtels au total**
- **Sortie** : `hotels_multi_cities.csv`

### 7. Fusion météo + hôtels
- Jointure (`merge`) sur la colonne `city`
- Chaque hôtel hérite des indicateurs météo de sa ville
- **Sortie** : `final_kayak_results.csv` — table centrale du projet (50 lignes × 12 colonnes)

### 8. Stockage cloud
- **Amazon S3** : stockage des fichiers bruts et traités (raw / processed / final)
- **Neon PostgreSQL** : chargement de la table `kayak_results` pour des requêtes SQL

### 9. Visualisation — Folium
- **Carte 1** : Top 5 destinations avec indicateurs météo au clic
- **Carte 2** : Top 20 hôtels numérotés par rang, colorés par ville, avec panneau liste latéral

---

## Résultats obtenus

| Étape | Résultat |
|---|---|
| Villes géocodées | 35 / 35 |
| Villes avec météo | 35 / 35 |
| Destinations sélectionnées | 5 |
| Hôtels scrapés | 50 (10 par ville) |
| Lignes en base SQL | 50 |
| Cartes générées | 2 |

---

## Choix techniques justifiés

| Choix | Justification |
|---|---|
| Nominatim plutôt que Google Maps | Gratuit, pas de clé requise, couverture France suffisante |
| `/forecast` plutôt que `/onecall` | Plan gratuit OpenWeather, données suffisantes pour 5 jours |
| ScrapingBee pour Booking | Booking bloque les requêtes directes, ScrapingBee gère le JS et les proxies |
| BeautifulSoup + `data-testid` | Sélecteur stable, indépendant des classes CSS qui changent régulièrement |
| Neon PostgreSQL | Base PostgreSQL hébergée dans le cloud, connexion simple via URL |
| Folium pour les cartes | Librairie Python native, export HTML standalone sans serveur |

---

## Limites identifiées

- Le score météo repose sur 5 jours de prévision (données du moment, pas historiques)
- Les coordonnées GPS des hôtels sur la carte sont approximées au centre de la ville
- Le scraping Booking peut être instable selon les variantes d'URL et les mises à jour du site
- La clé OpenWeather gratuite ne donne pas accès aux données historiques ni aux prévisions à 7 jours